In [2]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from matplotlib import cm, colors as mcolors

# ============================================================
# INPUT
# ============================================================
csv_file = "coast_xchange_skills_matrix_v6_institutions.csv"
df = pd.read_csv(csv_file)

# ============================================================
# AJUSTES PRINCIPALES DE TAMAÑO
# ============================================================
FIG_W = 1600
FIG_H = 1050

# IMPORTANTE:
# En Plotly NO puedes fijar manualmente el grosor de cada anillo por separado.
# Lo que sí puedes controlar es el tamaño TOTAL de la rueda dentro del canvas.
SUN_DOMAIN_X = [0.02, 0.62]
SUN_DOMAIN_Y = [0.08, 0.92]

# margen derecho para la leyenda
RIGHT_MARGIN = 470

# ============================================================
# ORDENES FIJOS A PARTIR DEL CSV
# ============================================================
spatial_order = (
    df[["spatial_order", "spatial_scale"]]
    .drop_duplicates()
    .sort_values("spatial_order")["spatial_scale"]
    .tolist()
)

family_order = (
    df[["family_order", "skill_family"]]
    .drop_duplicates()
    .sort_values("family_order")["skill_family"]
    .tolist()
)

environment_order = (
    df[["environment_order", "environment"]]
    .drop_duplicates()
    .sort_values("environment_order")["environment"]
    .tolist()
)

# ============================================================
# COLORES
# ============================================================
def make_palette(items, cmap_name, vmin=0.35, vmax=0.90):
    cmap = cm.get_cmap(cmap_name)
    vals = np.linspace(vmin, vmax, max(len(items), 1))
    return {item: mcolors.to_hex(cmap(v)) for item, v in zip(items, vals)}

# Si quieres cambiar colores de spatial, hazlo aquí
default_spatial_colors = {
    "local": "#b5d5ec",
    "subregion": "#4f93c2",
    "regional": "#155a9c",
}
# si hay alguna categoría nueva, la rellena con una paleta azul
missing_spatial = [x for x in spatial_order if x not in default_spatial_colors]
if missing_spatial:
    extra = make_palette(missing_spatial, "Blues", 0.35, 0.85)
    default_spatial_colors.update(extra)
spatial_colors = {k: default_spatial_colors[k] for k in spatial_order}

family_colors = make_palette(family_order, "Greens", 0.35, 0.85)
environment_colors = make_palette(environment_order, "Pastel2", 0.20, 0.95)

# anillo exterior completamente blanco
OUTER_RING_COLOR = "#FFFFFF"

# ============================================================
# MAPAS display -> texto visible
# ============================================================
if "short_code" not in df.columns:
    if "outer_code_display" in df.columns:
        df["short_code"] = df["outer_code_display"]
    else:
        raise ValueError("No encuentro ni 'short_code' ni 'outer_code_display' en el CSV.")

if "skill_family_display" not in df.columns:
    raise ValueError("Falta la columna 'skill_family_display' en el CSV.")
if "environment_display" not in df.columns:
    raise ValueError("Falta la columna 'environment_display' en el CSV.")

family_display_map = (
    df[["skill_family", "skill_family_display"]]
    .drop_duplicates()
    .set_index("skill_family")["skill_family_display"]
    .to_dict()
)

environment_display_map = (
    df[["environment", "environment_display"]]
    .drop_duplicates()
    .set_index("environment")["environment_display"]
    .to_dict()
)

# Para hover: el short_code apunta al skill_name completo
code_to_skill = (
    df[["short_code", "skill_name"]]
    .drop_duplicates()
    .set_index("short_code")["skill_name"]
    .to_dict()
)

# ============================================================
# TREE BUILDER
# ============================================================
def build_tree(df, root_id="ROOT"):
    # columnas jerárquicas visibles
    full_cols = [
        "skill_family",
        "environment",
        "spatial_scale",
        "short_code",   # anillo externo blanco
    ]

    leaves = (
        df[full_cols + ["node_value"]]
        .dropna(subset=full_cols)
        .groupby(full_cols, as_index=False, observed=True)["node_value"]
        .sum()
        .rename(columns={"node_value": "value"})
    )

    nodes = []
    total_value = leaves["value"].sum()

    nodes.append({
        "id": root_id,
        "parent": "",
        "depth": 0,
        "value": total_value,
        "display_text": "",
        "color": "rgba(255,255,255,0)",
        "full_family": "",
        "full_skill": "",
        "full_environment": "",
        "full_spatial": "",
        "full_code": "",
    })

    for depth in range(1, 5):
        group_cols = full_cols[:depth]

        agg = (
            leaves[group_cols + ["value"]]
            .groupby(group_cols, as_index=False, observed=True)["value"]
            .sum()
        )

        for _, row in agg.iterrows():
            vals = [str(row[c]) for c in group_cols]

            node_id = " | ".join(vals)
            parent_id = root_id if depth == 1 else " | ".join(vals[:-1])

            family = vals[0] if len(vals) >= 2 else ""
            env = vals[1] if len(vals) >= 3 else ""
            spatial = vals[2] if len(vals) >= 4 else ""
            code = vals[3] if len(vals) >= 5 else ""
            skill = code_to_skill.get(code, "")

            # texto visible por anillo
            if depth == 4:
                leaf_match = leaves[
                    (leaves["skill_family"] == family) &
                    (leaves["environment"] == env) &
                    (leaves["spatial_scale"] == spatial) &
                    (leaves["short_code"] == code)
                ]
                skill = leaf_match["full_skill"].iloc[0] if not leaf_match.empty else ""
            else:
                skill = ""

            if  depth == 1:
                display_text = family_display_map[family]
                color = family_colors[family]
            elif depth == 2:
                display_text = environment_display_map[env]
                color = environment_colors[env]
            elif depth == 3:
                display_text = spatial
                color = spatial_colors.get(spatial, "#4f93c2")
            elif depth == 4:
                display_text = code
                color = OUTER_RING_COLOR

            nodes.append({
                "id": node_id,
                "parent": parent_id,
                "depth": depth,
                "value": row["value"],
                "display_text": display_text,
                "color": color,
                "full_spatial": spatial,
                "full_family": family,
                "full_skill": skill,
                "full_environment": env,
                "full_code": code,
            })

    return pd.DataFrame(nodes)

nodes = build_tree(df)

# ============================================================
# FIGURE
# ============================================================
fig = go.Figure(go.Sunburst(
    ids=nodes["id"],
    parents=nodes["parent"],
    values=nodes["value"],
    branchvalues="total",
    sort=False,

    labels=nodes["display_text"],
    textinfo="label",

    customdata=np.stack([
        nodes["full_family"],
        nodes["full_skill"],
        nodes["full_environment"],
        nodes["full_spatial"],
        nodes["full_code"],
    ], axis=-1),

    insidetextorientation="radial",

    marker=dict(
        colors=nodes["color"],
        line=dict(color="black", width=1.25)
    ),

    root=dict(color="rgba(255,255,255,0)")
))

fig.update_traces(
    insidetextfont=dict(
        size=14,
        family="Arial Black, Arial, sans-serif",
        color="black"
    ),
    hovertemplate=(
        "<b>Skill family:</b> %{customdata[1]}<br>"
        "<b>Skill name:</b> %{customdata[2]}<br>"
        "<b>Environment:</b> %{customdata[3]}<br>"
        "<b>Spatial scale:</b> %{customdata[0]}<br>"
        "<b>Code:</b> %{customdata[4]}<extra></extra>"
    ),
    domain=dict(x=SUN_DOMAIN_X, y=SUN_DOMAIN_Y)
)

fig.update_layout(
    width=FIG_W,
    height=FIG_H,
    margin=dict(t=30, l=30, r=RIGHT_MARGIN, b=30),
    paper_bgcolor="white",
    plot_bgcolor="white",
    uniformtext=dict(minsize=10, mode="hide")
)

# ============================================================
# LEYENDA: color -> label completo
# ============================================================
def add_color_legend_block(fig, title, items, x0, y0, dy=0.030, box_w=0.018, box_h=0.018,
                           title_size=16, text_size=14):
    fig.add_annotation(
        x=x0, y=y0,
        xref="paper", yref="paper",
        text=f"<b>{title}</b>",
        showarrow=False,
        xanchor="left",
        yanchor="top",
        align="left",
        font=dict(size=title_size, color="black", family="Arial")
    )

    y = y0 - 0.035
    for color, label in items:
        fig.add_shape(
            type="rect",
            xref="paper", yref="paper",
            x0=x0, x1=x0 + box_w,
            y0=y - box_h + 0.004, y1=y + 0.004,
            line=dict(color="black", width=0.8),
            fillcolor=color
        )

        fig.add_annotation(
            x=x0 + box_w + 0.010, y=y,
            xref="paper", yref="paper",
            text=label,
            showarrow=False,
            xanchor="left",
            yanchor="middle",
            align="left",
            font=dict(size=text_size, color="black", family="Arial")
        )
        y -= dy

family_items = [(family_colors[k], k) for k in family_order]
environment_items = [(environment_colors[k], k) for k in environment_order]

add_color_legend_block(fig, "Skill family", family_items, x0=0.68, y0=0.96, dy=0.040)
add_color_legend_block(fig, "Environment", environment_items, x0=0.68, y0=0.68, dy=0.034)



fig.show()


C:\Users\freitasl\AppData\Local\Temp\ipykernel_573060\4224451968.py:55: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



KeyError: ''

In [5]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from matplotlib import cm, colors as mcolors
from PIL import Image

# ============================================================
# INPUT
# ============================================================
csv_file = "./data/coast_xchange_skills_matrix_v6_institutions.csv"
df = pd.read_csv(csv_file)

# ============================================================
# AJUSTES PRINCIPALES DE TAMAÑO
# ============================================================
FIG_W = 1750
FIG_H = 1100
SUN_DOMAIN_X = [0.00, 0.70]
SUN_DOMAIN_Y = [0.02, 0.98]
RIGHT_MARGIN = 500

# ============================================================
# ORDENES FIJOS
# ============================================================
institution_order = (
    df[["institution_order", "institution"]]
    .drop_duplicates()
    .sort_values("institution_order")["institution"]
    .tolist()
)

institution_full_name_map = (
    df[["institution", "full_name"]]
    .drop_duplicates()
    .set_index("institution")["full_name"]
    .to_dict()
)

spatial_order = (
    df[["spatial_order", "spatial_scale"]]
    .drop_duplicates()
    .sort_values("spatial_order")["spatial_scale"]
    .tolist()
)

family_order = (
    df[["family_order", "skill_family"]]
    .drop_duplicates()
    .sort_values("family_order")["skill_family"]
    .tolist()
)

environment_order = (
    df[["environment_order", "environment"]]
    .drop_duplicates()
    .sort_values("environment_order")["environment"]
    .tolist()
)

# ============================================================
# COLORES
# ============================================================
def make_palette(items, cmap_name, vmin=0.30, vmax=0.90):
    cmap = cm.get_cmap(cmap_name)
    vals = np.linspace(vmin, vmax, max(len(items), 1))
    return {item: mcolors.to_hex(cmap(v)) for item, v in zip(items, vals)}

institution_colors = make_palette(institution_order, "tab20", 0.00, 1.00)

# Reuse the current palette logic from your reduced CSV
spatial_colors = {
    "local": "#b5d5ec",
    "subregion": "#4f93c2",
    "regional": "#155a9c",
}

family_colors = make_palette(family_order, "Greens", 0.35, 0.85)
environment_colors = make_palette(environment_order, "Pastel2", 0.20, 0.95)
OUTER_RING_COLOR = "#FFFFFF"

# ============================================================
# MAPAS display -> texto visible
# ============================================================
if "short_code" not in df.columns:
    if "outer_code_display" in df.columns:
        df["short_code"] = df["outer_code_display"]
    else:
        raise ValueError("No encuentro ni 'short_code' ni 'outer_code_display' en el CSV.")

if "skill_family_display" not in df.columns:
    raise ValueError("Falta la columna 'skill_family_display' en el CSV.")
if "environment_display" not in df.columns:
    raise ValueError("Falta la columna 'environment_display' en el CSV.")

family_display_map = (
    df[["skill_family", "skill_family_display"]]
    .drop_duplicates()
    .set_index("skill_family")["skill_family_display"]
    .to_dict()
)

environment_display_map = (
    df[["environment", "environment_display"]]
    .drop_duplicates()
    .set_index("environment")["environment_display"]
    .to_dict()
)

# ============================================================
# TREE BUILDER
# ============================================================
def build_tree(df, root_id="ROOT"):
    # jerarquía visible real
    full_cols = [
        "institution",
        "skill_family",
        "environment",
        "spatial_scale",
        "short_code",
    ]

    # hojas: una fila por combinación visible
    leaves = (
        df[full_cols + ["node_value", "skill_name"]]
        .dropna(subset=full_cols)
        .groupby(full_cols, as_index=False, observed=True)
        .agg(
            value=("node_value", "sum"),
            full_skill=("skill_name", lambda s: " / ".join(sorted(set(map(str, s)))))
        )
    )

    nodes = []
    total_value = leaves["value"].sum()

    nodes.append({
        "id": root_id,
        "parent": "",
        "depth": 0,
        "value": total_value,
        "display_text": "",
        "color": "rgba(255,255,255,0)",
        "full_institution": "",
        "full_family": "",
        "full_environment": "",
        "full_spatial": "",
        "full_code": "",
    })

    for depth in range(1, 6):
        group_cols = full_cols[:depth]

        # agregación correcta por nivel
        agg = (
            leaves[group_cols + ["value"]]
            .groupby(group_cols, as_index=False, observed=True)["value"]
            .sum()
        )

        for _, row in agg.iterrows():
            vals = [str(row[c]) for c in group_cols]

            node_id = " | ".join(vals)
            parent_id = root_id if depth == 1 else " | ".join(vals[:-1])

            inst = vals[0] if len(vals) >= 1 else ""
            family = vals[1] if len(vals) >= 2 else ""
            env = vals[2] if len(vals) >= 3 else ""
            spatial = vals[3] if len(vals) >= 4 else ""
            code = vals[4] if len(vals) >= 5 else ""

            # skill_name solo en hojas
            if depth == 5:
                leaf_match = leaves[
                    (leaves["institution"] == inst) &
                    (leaves["skill_family"] == family) &
                    (leaves["environment"] == env) &
                    (leaves["spatial_scale"] == spatial) &
                    (leaves["short_code"] == code)
                ]
                skill = leaf_match["full_skill"].iloc[0] if not leaf_match.empty else ""
            else:
                skill = ""

            if depth == 1:
                display_text = inst
                color = institution_colors.get(inst, "#BDBDBD")
            elif depth == 2:
                display_text = family_display_map[family]
                color = family_colors[family]
            elif depth == 3:
                display_text = environment_display_map[env]
                color = environment_colors[env]
            elif depth == 4:
                display_text = spatial
                color = spatial_colors.get(spatial, "#4f93c2")
            elif depth == 5:
                display_text = code
                color = OUTER_RING_COLOR

            nodes.append({
                "id": node_id,
                "parent": parent_id,
                "depth": depth,
                "value": row["value"],
                "display_text": display_text,
                "color": color,
                "full_institution": inst,
                "full_family": family,
                "full_environment": env,
                "full_spatial": spatial,
                "full_code": code,
            })

    return pd.DataFrame(nodes)

nodes = build_tree(df)

# ============================================================
# FIGURE
# ============================================================
fig = go.Figure(go.Sunburst(
    ids=nodes["id"],
    parents=nodes["parent"],
    values=nodes["value"],
    branchvalues="total",
    sort=False,
    labels=nodes["display_text"],
    textinfo="label",
    customdata=np.stack([
        nodes["full_institution"],
        nodes["full_family"],
        nodes["full_environment"],
        nodes["full_spatial"],
        nodes["full_code"],
    ], axis=-1),
    insidetextorientation="radial",
    marker=dict(
        colors=nodes["color"],
        line=dict(color="black", width=0)
    ),
    root=dict(color="rgba(255,255,255,0)")
))


fig.update_traces(
    insidetextfont=dict(
        size=10,
        family="Arial Black, Arial, sans-serif",
        color="black"
    ),
    hovertemplate=(
        "<b>Institution:</b> %{customdata[0]}<br>"
        "<b>Skill family:</b> %{customdata[1]}<br>"
        "<b>Environment:</b> %{customdata[2]}<br>"
        "<b>Spatial scale:</b> %{customdata[3]}<br>"
        "<b>Code:</b> %{customdata[4]}<extra></extra>"
    ),
    domain=dict(x=SUN_DOMAIN_X, y=SUN_DOMAIN_Y)
)

fig.update_layout(
    width=FIG_W,
    height=FIG_H,
    margin=dict(t=5, l=5, r=RIGHT_MARGIN, b=5, pad=0),
    paper_bgcolor="white",
    plot_bgcolor="white",
    uniformtext=dict(minsize=9, mode="hide")
)

# ============================================================
# LEYENDAS
# ============================================================
def add_color_legend_block(fig, title, items, x0, y0, dy=0.028, box_w=0.016, box_h=0.016,
                           title_size=15, text_size=14):
    fig.add_annotation(
        x=x0, y=y0, xref="paper", yref="paper",
        text=f"<b>{title}</b>", showarrow=False,
        xanchor="left", yanchor="top", align="left",
        font=dict(size=title_size, color="black", family="Arial")
    )
    y = y0 - 0.032
    for color, label in items:
        fig.add_shape(
            type="rect", xref="paper", yref="paper",
            x0=x0, x1=x0 + box_w,
            y0=y - box_h + 0.004, y1=y + 0.004,
            line=dict(color="black", width=0.8),
            fillcolor=color
        )
        fig.add_annotation(
            x=x0 + box_w + 0.010, y=y, xref="paper", yref="paper",
            text=label, showarrow=False,
            xanchor="left", yanchor="middle", align="left",
            font=dict(size=text_size, color="black", family="Arial")
        )
        y -= dy

spatial_items = [(spatial_colors[k], k) for k in spatial_order if k in spatial_colors]
family_items = [(family_colors[k], k) for k in family_order]
environment_items = [(environment_colors[k], k) for k in environment_order]
institution_items = [
    (institution_colors[k], institution_full_name_map.get(k, k))
    for k in institution_order
    if k in institution_colors
]

add_color_legend_block(fig, "Spatial scale", spatial_items, x0=0.70, y0=0.75, dy=0.033)
add_color_legend_block(fig, "Skill family", family_items, x0=0.70, y0=0.62, dy=0.030)
add_color_legend_block(fig, "Environment", environment_items, x0=0.70, y0=0.45, dy=0.030)
add_color_legend_block(
    fig,
    "Institution",
    institution_items,
    x0=0.92,
    y0=0.76,
    dy=0.028,
    box_w=0.014,
    box_h=0.014,
    title_size=15,
    text_size=13,
)

# ============================================================
# LOGO EN EL CENTRO
# ============================================================
logo = Image.open("./assets/LOGO.png")

# centro automático de la rueda
cx = (SUN_DOMAIN_X[0] + SUN_DOMAIN_X[1]) / 2
cy = (SUN_DOMAIN_Y[0] + SUN_DOMAIN_Y[1]) / 2

# tamaño del logo y del círculo blanco de apoyo
logo_w = 0.12
logo_h = 0.14
bg_r_x = 0.09
bg_r_y = 0.10

fig.add_shape(
    type="circle",
    xref="paper", yref="paper",
    x0=cx - bg_r_x, x1=cx + bg_r_x,
    y0=cy - bg_r_y, y1=cy + bg_r_y,
    fillcolor=None,
    line=dict(color="rgba(0,0,0,0)", width=0),
    layer="below"
)

fig.add_layout_image(
    dict(
        source=logo,
        xref="paper",
        yref="paper",
        x=cx,
        y=cy,
        sizex=logo_w,
        sizey=logo_h,
        xanchor="center",
        yanchor="middle",
        sizing="contain",
        layer="below",
        opacity=1.0
    )
)


fig.write_html(
    "./assets/interactive_secondment_chart.html",
    include_plotlyjs="cdn",
    full_html=True,
    default_width="100%",
    default_height="100%"
)

fig.show()


C:\Users\freitasl\AppData\Local\Temp\ipykernel_508604\3814124416.py:64: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.



In [5]:
spatial_order

['site', 'subregion', 'regional']